In [ ]:
from google.colab import userdata
openai_key = userdata.get('OPENAI_API_KEY')
google_key = userdata.get('GOOGLE_API_KEY')
pinecone_api_key = userdata.get('PINECONE_API_KEY')

### MOUNTING YOUR GOOGLE DRIVE IN THE CURRENT PROJECT

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
# Define your data path
data_dir = "/content/drive/MyDrive/Session 3 Rag/data"

In [ ]:
import os
os.getcwd()
os.listdir(data_dir)

['personal_loan.txt',
 'two_wheeler_loan.txt',
 'no_cost_emi.txt',
 'home_loan.txt',
 'FAQs.txt',
 'eligibility_documents.txt',
 'loan_against_property.txt',
 'about_us.txt']

In [ ]:
!pip install pinecone

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 742.7/742.7 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.9/280.9 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 3.8 MB/s eta 0:00:00
  Attempting uninstall: packaging
    Found existing installation: packaging 26.0
    Uninstalling packaging-26.0:
      Successfully uninstalled packaging-26.0


In [ ]:
!pip install sentence-transformers

### Step 1: Access your Vector DB remotely

In [ ]:
from pinecone import Pinecone, ServerlessSpec
import time

pc = Pinecone(api_key=pinecone_api_key)
# I want to meet vector-hybrid
index_name = 'vector-hybrid'

### Step 2: Check if your index exists, if not create it.

In [ ]:
pc.list_indexes()

[]

In [ ]:
# Step 2. Check if index exists, if not, create it
if index_name not in [idx.name for idx in pc.list_indexes()]:
    print(f"Index '{index_name}' not found. Creating it...")
    pc.create_index(
        name=index_name,
        dimension=384, # Must match paraphrase-MiniLM-L6-v2
        metric='cosine',
        spec=ServerlessSpec(cloud='aws', region='us-east-1')
    )
    # Optional
    # Wait for index to spin up
    while not pc.describe_index(index_name).status['ready']:
        time.sleep(10)
    print("Index created and ready.")
else:
    print(f"Index '{index_name}' already exists.")

# Connect to the index
index = pc.Index(index_name)

Index 'vector-hybrid' not found. Creating it...
Index created and ready.


### Step 3 - Get your Documents in one place in an organized format

In [ ]:
# WHatever documents you have read them and organize them like this
# You'll first read your docs - function to read docs
# You'll organize them like the below.
docs = [
    {"id": "doc1", "text": "Pandas is a Python library for data analysis."},
    {"id": "doc2", "text": "Pinecone is a vector database for semantic search."},
    {"id": "doc3", "text": "Spark enables distributed data processing."},
]

### Step 4 - Creating Embedding

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("paraphrase-MiniLM-L6-v2")  # 384-dim
embeddings = model.encode([d["text"] for d in docs])  # shape (n, 384)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/314 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
embeddings[0].shape

(384,)

### Step 5 - Organize Docs and Embeddings together

In [ ]:
# {id: 1, values: [0.03, -0.04, 0.95,.....], metatadata: {"text":"Pandas is a Python library for data analysis"}}

In [ ]:
vectors = [
    {
        "id": d["id"],
        "values": emb.tolist(),
        "metadata": {"text": d["text"]},
    }
    for d, emb in zip(docs, embeddings)
]

In [ ]:
vectors[0]

{'id': 'doc1',
 'values': [-0.48840245604515076,
  -0.7727128863334656,
  -0.1793675422668457,
  0.275081068277359,
  0.10556235164403915,
  -0.2619451582431793,
  0.5112577080726624,
  -0.12202758342027664,
  0.1513729691505432,
  0.4513612985610962,
  -0.18327559530735016,
  -0.38937461376190186,
  0.21370477974414825,
  0.3723406493663788,
  -0.4156704246997833,
  -0.02121276967227459,
  0.34635457396507263,
  -0.05766167864203453,
  -0.028359429910779,
  -0.5018450617790222,
  -0.4719994068145752,
  -0.05063016340136528,
  -0.225893035531044,
  0.38802066445350647,
  -0.21016232669353485,
  -0.5842373967170715,
  -0.2579244077205658,
  -0.04914756491780281,
  0.06051091477274895,
  0.04013308137655258,
  0.14206963777542114,
  -0.23850703239440918,
  -0.06560961902141571,
  0.3726140558719635,
  -0.43465307354927063,
  0.2550434172153473,
  -0.051754217594861984,
  -0.3306208550930023,
  -0.12578822672367096,
  0.13709743320941925,
  -0.3844870328903198,
  0.3623211085796356,
  0.6

### Step 6 - Push the above organized combination to Vector DB

In [ ]:
index.upsert(vectors)

UpsertResponse(upserted_count=3, _response_info={'raw_headers': {'date': 'Thu, 12 Mar 2026 06:40:49 GMT', 'content-type': 'application/json', 'content-length': '19', 'connection': 'keep-alive', 'x-pinecone-request-lsn': '1', 'x-pinecone-request-logical-size': '4793', 'x-pinecone-request-latency-ms': '339', 'x-envoy-upstream-service-time': '323', 'x-pinecone-response-duration-ms': '342', 'grpc-status': '0', 'server': 'envoy'}})

### Upserting documents in the Vector DB


In [ ]:
import os
os.listdir(data_dir)

['personal_loan.txt',
 'two_wheeler_loan.txt',
 'no_cost_emi.txt',
 'home_loan.txt',
 'FAQs.txt',
 'eligibility_documents.txt',
 'loan_against_property.txt',
 'about_us.txt']

### Step 1: Access your vector DB remotely

In [ ]:
from pinecone import Pinecone, ServerlessSpec
import time

pc = Pinecone(api_key=pinecone_api_key)
# I want to meet vector-hybrid
index_name = 'vector-docs'

### Step 2: Check if Index exists else create it

In [ ]:
# Step 2. Check if index exists, if not, create it
if index_name not in [idx.name for idx in pc.list_indexes()]:
    print(f"Index '{index_name}' not found. Creating it...")
    pc.create_index(
        name=index_name,
        dimension=384, # Must match paraphrase-MiniLM-L6-v2
        metric='cosine',
        spec=ServerlessSpec(cloud='aws', region='us-east-1')
    )
    # Optional
    # Wait for index to spin up
    while not pc.describe_index(index_name).status['ready']:
        time.sleep(10)
    print("Index created and ready.")
else:
    print(f"Index '{index_name}' already exists.")

# Connect to the index
index = pc.Index(index_name)

Index 'vector-docs' not found. Creating it...
Index created and ready.


### Step 3: Organize you DOCS

In [ ]:
def simple_chunker(text, size = 500):
  return [text[i:i+size] for i in range(0, len(text), size)]

In [ ]:
print(data_dir)

/content/drive/MyDrive/Session 3 Rag/data


In [ ]:
# One file -
docs = []
# Changed os.listdir() to os.listdir(data_dir) to get files from the correct directory
# Picking the first file from the data_dir for demonstration purposes.
filename = os.listdir(data_dir)[0]
print(filename)
with open(os.path.join(data_dir, filename), "r", encoding="utf-8") as f:
  content = f.read()
  chunks = simple_chunker(content)
  for i, chunk in enumerate(chunks):
    docs.append({"id": f"{filename}_chunk_{i}",
                "text": chunk
    })


personal_loan.txt


In [ ]:
## All the files
docs = []
# filename = os.listdir(data_dir)[0]
for filename in os.listdir(data_dir):
  print(filename)
  if filename.endswith(".txt"):
    # Replace this reading part with a function to read from html/word/pdf
    with open(os.path.join(data_dir, filename), "r", encoding="utf-8") as f:
      content = f.read()
      chunks = simple_chunker(content)
      for i, chunk in enumerate(chunks):
        docs.append({"id": f"{filename}_chunk_{i}",
                    "text": chunk
      })

personal_loan.txt
two_wheeler_loan.txt
no_cost_emi.txt
home_loan.txt
FAQs.txt
eligibility_documents.txt
loan_against_property.txt
about_us.txt


In [ ]:
print(f"Total chunks are {len(docs)}")
docs[:5]

Total chunks are 74


[{'id': 'personal_loan.txt_chunk_0',
  'text': 'BrightBridge Finance — Personal Loan\nSimple loans. Clear terms. Fast decisions.\n\nOverview\nA Personal Loan is an unsecured loan designed for quick access to funds without the need to pledge collateral. It is suitable when you need flexible financing for planned or unplanned expenses—medical needs, education, travel, wedding expenses, home improvement, or consolidating multiple debts into a single EMI.\n\nWhy customers choose our Personal Loan\n• Fast in-principle decision for many profiles once key '},
 {'id': 'personal_loan.txt_chunk_1',
  'text': 'details are verified\n• No collateral required (unsecured)\n• Flexible tenors and EMI options\n• Transparent communication via Key Fact Statement (KFS) and repayment schedule\n• Digital-first experience (where available), with offline support when needed\n\nTypical use cases\nDebt consolidation: Convert multiple high-interest dues into one structured EMI.\nEmergency expenses: Medical proced

### Step 4 - Create Embeddings

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("paraphrase-MiniLM-L6-v2")  # 384-dim
embeddings = model.encode([d["text"] for d in docs])  # shape (n, 384)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### Step 5 - Organize Docs & Embeddings together

In [ ]:
vectors = [
    {
        "id": d["id"],
        "values": emb.tolist(),
        "metadata": {"text": d["text"]},
    }
    for d, emb in zip(docs, embeddings)
]

In [ ]:
vectors[1]

{'id': 'personal_loan.txt_chunk_1',
 'values': [-0.018106136471033096,
  -0.0229153111577034,
  -0.1402844488620758,
  -0.10296934843063354,
  0.07184197753667831,
  0.10295578092336655,
  0.0708305835723877,
  0.11348114162683487,
  0.07362773269414902,
  -0.06046473979949951,
  0.1734171062707901,
  -0.06294803321361542,
  -0.18183006346225739,
  -0.20512162148952484,
  0.02821555733680725,
  -0.07241377979516983,
  -0.06265410035848618,
  -0.03949782997369766,
  0.022339114919304848,
  0.08587708324193954,
  0.08160178363323212,
  -0.05607891455292702,
  -0.30021488666534424,
  0.06351438909769058,
  0.2656008005142212,
  0.22057993710041046,
  0.11624772101640701,
  -0.06481040269136429,
  0.06036270782351494,
  -0.33906790614128113,
  0.18636733293533325,
  0.05440293997526169,
  0.20469063520431519,
  0.01586117409169674,
  0.41667038202285767,
  -0.18454720079898834,
  -0.11988432705402374,
  0.16771863400936127,
  0.0790591612458229,
  -0.2863398790359497,
  0.02227133885025978

### Step 6 - Push the above organized combination to Vector DB

In [ ]:
index.upsert(vectors)

UpsertResponse(upserted_count=74, _response_info={'raw_headers': {'date': 'Thu, 12 Mar 2026 06:50:47 GMT', 'content-type': 'application/json', 'content-length': '20', 'connection': 'keep-alive', 'x-pinecone-request-lsn': '1', 'x-pinecone-request-logical-size': '151782', 'x-pinecone-request-latency-ms': '423', 'x-envoy-upstream-service-time': '327', 'x-pinecone-response-duration-ms': '425', 'grpc-status': '0', 'server': 'envoy'}})

### Query these docs

In [ ]:
# 1. Define the user's question
query_text = "How do I get a two wheeler loan?"

# 2. Convert query to vector (using the same model from Step 4)
query_vector = model.encode([query_text])[0].tolist()

# 3. Query Pinecone
# top_k=3 returns the 3 most similar chunks
results = index.query(
    vector=query_vector,
    top_k=3,
    include_metadata=True
)

# 4. Display Results
print(f"Results for Query: '{query_text}'\n")
print("-" * 50)


for i, match in enumerate(results["matches"]):
    print(f"Match {i+1} (Score: {match['score']:.4f})")
    print(f"Source: {match['metadata'].get('source')}")
    print(f"Text Snippet: {match['metadata'].get('text')[:200]}...")
    print("-" * 50)

Results for Query: 'How do I get a two wheeler loan?'

--------------------------------------------------
Match 1 (Score: 0.5262)
Source: None
Text Snippet: BrightBridge Finance — Two-Wheeler Loan
Simple loans. Clear terms. Fast decisions.

Overview
A Two-Wheeler Loan helps you finance the purchase of a scooter or motorcycle through affordable monthly EMI...
--------------------------------------------------
Match 2 (Score: 0.4231)
Source: None
Text Snippet: ns.

5) Product-Specific FAQs

Personal Loan
Q18. What can I use a personal loan for?
A. Common uses include medical expenses, education, travel, home improvement, wedding expenses, debt consolidation...
--------------------------------------------------
Match 3 (Score: 0.4198)
Source: None
Text Snippet: er, subject to due diligence and property/legal checks.

Loan Against Property (LAP)
Q20. How is LAP different from a home loan?
A. LAP is a secured loan where you pledge an existing residential/comme...
---------------------------

In [ ]:
results

QueryResponse(matches=[{'id': 'two_wheeler_loan.txt_chunk_0',
 'metadata': {'text': 'BrightBridge Finance — Two-Wheeler Loan\n'
                      'Simple loans. Clear terms. Fast decisions.\n'
                      '\n'
                      'Overview\n'
                      'A Two-Wheeler Loan helps you finance the purchase of a '
                      'scooter or motorcycle through affordable monthly EMIs. '
                      'It is typically secured against the vehicle and is '
                      'designed to keep your upfront cost manageable while '
                      'enabling ownership sooner.\n'
                      '\n'
                      'Who this loan is for\n'
                      '• First-time buyers who want manageable EMIs\n'
                      '• Working professionals who need daily commuting '
                      'support\n'
                      '• Small business owners who need mobility f'},
 'score': 0.526211083,
 'values': []}, {'id': 'FAQs.